# 🏋️ Kaggle Training Pipeline — Business Entity Resolution

Runs the **full ML pipeline on Kaggle** using pre-cleaned datasets (no re-cleaning needed).

## 💾 Checkpoint Strategy

Every expensive stage saves its output to `/kaggle/working/output/` on Kaggle's disk.
After each save, a **⬇️ download button appears immediately in the cell output** — click it to download to your local machine right away without waiting for the notebook to finish.

| Checkpoint File | Saved After | Reuse when tuning? |
|---|---|---|
| `train_candidates.parquet` / `val_candidates.parquet` | Stage 4 (Blocking) | ✅ Reusable — skip if only changing features or hyperparams |
| `train_pairs.parquet` / `val_pairs.parquet` | Stage 6 (Hard Negatives) | ✅ Reusable — skip if only changing features or hyperparams |
| `train_features.parquet` / `val_features.parquet` | Stage 7 (Feature Extraction) | 🔥 Reusable — huge time saver when tuning XGBoost |
| `model.json` | Stage 8 (XGBoost Training) | ❌ Always re-train when tuning hyperparams |
| `test_candidates.parquet` | Stage 12 (Test Blocking) | ✅ Reusable — skip unless you changed blocking rules |
| `test_features.parquet` | Stage 13 (Test Features) | 🔥 Reusable — skip unless you added new features |
| `matching_results.tsv` / `candidate_pairs.tsv` | Stage 15 (Submission) | ❌ Re-generate after every threshold or model change |

| Stage | Description |
|-------|-------------|
| 1 | Clone repo & install deps |
| 2 | Config + PipelineState init + download helper |
| 3 | Load pre-cleaned datasets |
| 4 | Train / Val split |
| 5 | 8-pass TF-IDF & Token Blocking (+ checkpoint + ⬇️ download) |
| 6 | Blocking Recall evaluation |
| 7 | Hard-negative pair building (+ checkpoint + ⬇️ download) |
| 8 | 30+ pairwise feature extraction (+ checkpoint + ⬇️ download) |
| 9 | XGBoost training (GPU) |
| 10 | Validation scoring |
| 11 | Macro F0.5 threshold sweep |
| 12 | Error analysis |
| 13 | Test candidate generation (+ checkpoint + ⬇️ download) |
| 14 | Test feature extraction (+ checkpoint + ⬇️ download) |
| 15 | Test inference |
| 16 | Write submission TSVs (+ ⬇️ download) |
| 17 | Inspect output files |
| 18 | 📦 Download ALL checkpoints as one zip |


---
## 📦 Cell 1: Clone Repository & Install Dependencies

In [ ]:
import os

%cd /kaggle/working
!rm -rf ml_challenge
!git clone https://github.com/BharathPESU/ml_challenge.git
%cd /kaggle/working/ml_challenge/Error_404_Not_Found_submission/code/business_entity_resolution
!git checkout Bharath.2004
!git pull origin Bharath.2004
%pip install -r requirements.txt --quiet

print("\n✓ Repository cloned and dependencies installed!")
print("✓ CWD:", os.getcwd())


---
## ⚙️ Cell 2: Initialise Config, PipelineState & Download Helper

> **`_download_btn(*files)`** — A helper defined here that renders a styled **⬇️ clickable download button** inside any cell's output area. It is called automatically after each checkpoint save so you can download files immediately without waiting for the full pipeline to finish.

In [ ]:
import sys, os, logging
import pandas as pd
from IPython.display import display, HTML

SRC = "/kaggle/working/ml_challenge/Error_404_Not_Found_submission/code/business_entity_resolution/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from config import Config
from pipeline import PipelineState

# ── 🩺 SANITY CHECK / DRY RUN TOGGLE ─────────────────────────────────────
# Set SANITY_CHECK = True for a 10-second test run on 50 rows (verifies notebook end-to-end)
# Set SANITY_CHECK = False when you are ready to run the full dataset (1.7M rows)!
SANITY_CHECK = False

CLEANED_TRAIN = "/kaggle/input/datasets/cdbharath234/cleaned-data/cleaned"
CLEANED_TEST  = "/kaggle/input/datasets/cdbharath234/cleaned-data/cleaned"
GT_PATH       = "/kaggle/input/datasets/cdbharath234/train-ground-truth-ml-challenge/train_ground_truth.tsv"

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
CKPT_DIR   = OUTPUT_DIR

def _download_btn(*file_paths):
    """Render an immediate clickable download button for saved files."""
    buttons_html = []
    for fpath in file_paths:
        if not os.path.exists(fpath):
            continue
        fname = os.path.basename(fpath)
        rel   = os.path.relpath(fpath, "/kaggle/working")
        mb    = os.path.getsize(fpath) / 1_048_576
        buttons_html.append(
            f'<a href="{rel}" download="{fname}" '
            f'style="display:inline-block;margin:5px 6px 5px 0;padding:9px 20px;'
            f'background:#0ea5e9;color:#fff;font-weight:bold;font-size:14px;'
            f'border-radius:6px;text-decoration:none;'
            f'box-shadow:0 2px 6px rgba(0,0,0,.25);">'
            f'⬇️ {fname} ({mb:.1f} MB)</a>'
        )
    if not buttons_html:
        return
    html = (
        '<div style="margin:12px 0;padding:12px 16px;'
        'background:#f0f9ff;border-left:4px solid #0ea5e9;border-radius:6px;">'
        '<b style="font-size:15px;">📥 File(s) saved — click to download to your local machine NOW:</b><br/><br/>'
        + ''.join(buttons_html)
        + '</div>'
    )
    display(HTML(html))

cfg = Config()
cfg.data.train_source1_path       = os.path.join(CLEANED_TRAIN, "train_source1_clean.tsv")
cfg.data.train_source2_path       = os.path.join(CLEANED_TRAIN, "train_source2_clean.tsv")
cfg.data.train_source3_path       = os.path.join(CLEANED_TRAIN, "train_source3_clean.tsv")
cfg.data.train_ground_truth_path  = GT_PATH

cfg.data.test_source1_path        = os.path.join(CLEANED_TEST, "test_source1_clean.tsv")
cfg.data.test_source2_path        = os.path.join(CLEANED_TEST, "test_source2_clean.tsv")
cfg.data.test_source3_path        = os.path.join(CLEANED_TEST, "test_source3_clean.tsv")

cfg.data.output_dir               = OUTPUT_DIR
cfg.xgboost.tree_method           = "hist"
cfg.xgboost.device                = "cuda"

if SANITY_CHECK:
    cfg.runtime.debug_sample_size = 50
    print("🩺 [SANITY CHECK MODE ON] Dataset size limited to 50 rows for a quick 10s end-to-end test!")
else:
    cfg.runtime.debug_sample_size = None
    print("🚀 [FULL DATASET MODE ON] Processing complete 1.7M row dataset!")

cfg.configure_logging()
state = PipelineState()
print("✓ Config & PipelineState ready")


---
## 🔍 Cell 3: Verify All Input Files Exist

In [ ]:
files_to_check = [
    ("train_source1_clean", cfg.data.train_source1_path),
    ("train_source2_clean", cfg.data.train_source2_path),
    ("train_source3_clean", cfg.data.train_source3_path),
    ("train_ground_truth",  cfg.data.train_ground_truth_path),
    ("test_source1_clean",  cfg.data.test_source1_path),
    ("test_source2_clean",  cfg.data.test_source2_path),
    ("test_source3_clean",  cfg.data.test_source3_path),
]

all_ok = True
for name, path in files_to_check:
    if os.path.exists(path):
        mb = os.path.getsize(path) / 1_048_576
        print(f"  ✅ {name:<28s}  {mb:7.1f} MB   {path}")
    else:
        print(f"  ❌ MISSING: {name:<22s}  {path}")
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        "One or more input files are missing. "
        "Make sure the cleaned_data Kaggle dataset and the original dataset are both attached."
    )
print("\n✅ All input files found — proceeding!")


---
## 📥 Cell 4: Load Pre-Cleaned Training Datasets & Ground Truth

In [ ]:
from io_utils import read_tsv, validate_source_file, validate_ground_truth_file
from ground_truth import parse_ground_truth

nrows = cfg.runtime.debug_sample_size
print(f"⏳ Loading pre-cleaned training datasets{' (SANITY CHECK 50 ROWS)' if nrows else ''}...")

state.train_s1_clean = read_tsv(cfg.data.train_source1_path, nrows=nrows)
state.train_s2_clean = read_tsv(cfg.data.train_source2_path, nrows=nrows)
state.train_s3_clean = read_tsv(cfg.data.train_source3_path, nrows=nrows)

state.train_s1 = state.train_s1_clean
state.train_s2 = state.train_s2_clean
state.train_s3 = state.train_s3_clean

gt_df = read_tsv(cfg.data.train_ground_truth_path)
state.gt = parse_ground_truth(gt_df)

assert state.gt is not None
print("✅ Loaded pre-cleaned training data:")
print(f"   S1 clean : {len(state.train_s1_clean):>10,} rows")
print(f"   S2 clean : {len(state.train_s2_clean):>10,} rows")
print(f"   S3 clean : {len(state.train_s3_clean):>10,} rows")


---
## ✂️ Cell 5 — Stage 3: Train / Validation Split (80 / 20)

In [ ]:
from pipeline import split_training_validation

print("⏳ Stage 3: Splitting entities into train / val...")
split_training_validation(cfg, state)
assert state.train_s1_ids is not None and state.val_s1_ids is not None
print(f"✅ Stage 3 Complete — Train IDs: {len(state.train_s1_ids):,} | Val IDs: {len(state.val_s1_ids):,}")


---
## 🔍 Cell 6 — Stage 4: Generate Candidate Pairs (8-pass Blocking)

> 💾 **Checkpoint:** Saves `train_candidates.parquet` and `val_candidates.parquet` to `/kaggle/working/output/`.
> A **⬇️ download button** will appear in this cell's output immediately after saving.
>
> **Re-run only if you changed:** blocking rules (`name_top_k`, `address_top_k`, blocking passes in `config.py`).

In [ ]:
import time, os, gc
import pandas as pd
from blocking import (
    generate_exact_match_candidates,
    generate_numeric_address_candidates,
    generate_rare_token_candidates,
    generate_fuzzy_candidates,
)
from splitting import filter_source_by_s1_ids
from io_utils import read_tsv
from ground_truth import parse_ground_truth
from pipeline import split_training_validation

train_cands_ckpt = os.path.join(CKPT_DIR, "train_candidates.parquet")
val_cands_ckpt   = os.path.join(CKPT_DIR, "val_candidates.parquet")

if os.path.exists(train_cands_ckpt) and os.path.exists(val_cands_ckpt):
    print("⚡ [CHECKPOINT HIT] Loading candidates from Kaggle disk — skipping blocking...")
    state.train_candidates = pd.read_parquet(train_cands_ckpt)
    state.val_candidates   = pd.read_parquet(val_cands_ckpt)
    print(f"   Loaded train_candidates: {len(state.train_candidates):,} rows")
    print(f"   Loaded val_candidates  : {len(state.val_candidates):,} rows")
else:
    # ── 🛡️ Auto-Recovery: Ensure prerequisite data (Cells 4 & 5) are loaded in memory ──
    if state.train_s1_clean is None or state.train_s2_clean is None or state.train_s3_clean is None:
        nrows = cfg.runtime.debug_sample_size
        print(f"⏳ [AUTO-RECOVERY] Loading pre-cleaned training data{' (SANITY CHECK 50 ROWS)' if nrows else ''}...")
        state.train_s1_clean = read_tsv(cfg.data.train_source1_path, nrows=nrows)
        state.train_s2_clean = read_tsv(cfg.data.train_source2_path, nrows=nrows)
        state.train_s3_clean = read_tsv(cfg.data.train_source3_path, nrows=nrows)
        gt_df = read_tsv(cfg.data.train_ground_truth_path)
        state.gt = parse_ground_truth(gt_df)
        state.train_s1 = state.train_s1_clean
        state.train_s2 = state.train_s2_clean
        state.train_s3 = state.train_s3_clean
    
    if state.train_s1_ids is None or state.val_s1_ids is None:
        print("⏳ [AUTO-RECOVERY] Running Stage 3 (Train / Val Split)...")
        split_training_validation(cfg, state)
        
    print("⏳ Stage 4: Resumable Vectorized 8-Pass Candidate Generation starting...\n")
    
    train_s1_df = filter_source_by_s1_ids(state.train_s1_clean, state.train_s1_ids)
    val_s1_df   = filter_source_by_s1_ids(state.train_s1_clean, state.val_s1_ids)
    
    def run_pass_incremental_resumable(s1_df, name_prefix):
        passes = [
            (1, "Exact Name Norm",       lambda s1, tgt, lbl: generate_exact_match_candidates(s1, tgt, "business_name_norm", lbl, 2)),
            (2, "Exact Core Name",       lambda s1, tgt, lbl: generate_exact_match_candidates(s1, tgt, "business_name_core", lbl, 3)),
            (3, "Exact Address Norm",    lambda s1, tgt, lbl: generate_exact_match_candidates(s1, tgt, "business_address_norm", lbl, 5)),
            (4, "Numeric Address",       lambda s1, tgt, lbl: generate_numeric_address_candidates(s1, tgt, lbl)),
            (5, "Rare Name Tokens",      lambda s1, tgt, lbl: generate_rare_token_candidates(s1, tgt, "business_name_norm", lbl, 100)),
            (6, "Rare Address Tokens",   lambda s1, tgt, lbl: generate_rare_token_candidates(s1, tgt, "business_address_norm", lbl, 100)),
            (7, "Fuzzy Name (RapidFuzz)", lambda s1, tgt, lbl: generate_fuzzy_candidates(s1, tgt, "business_name_norm", lbl, limit=10)),
            (8, "Fuzzy Address (RapidFuzz)", lambda s1, tgt, lbl: generate_fuzzy_candidates(s1, tgt, "business_address_norm", lbl, limit=10)),
        ]
        
        accumulated_df = pd.DataFrame()
        start_pass_idx = 0
        
        # ── 🔍 Smart Resume Check: Find latest completed pass on disk ─────────
        for p_num in range(8, 0, -1):
            pass_ckpt_path = os.path.join(CKPT_DIR, f"{name_prefix}_candidates_pass{p_num}.parquet")
            if os.path.exists(pass_ckpt_path):
                try:
                    accumulated_df = pd.read_parquet(pass_ckpt_path)
                    start_pass_idx = p_num  # resume after this pass
                    print(f"  ⚡ [RESUME HIT] Found existing Pass {p_num} checkpoint on disk ({len(accumulated_df):,} candidates)!")
                    print(f"     Skipping Passes 1–{p_num} and resuming directly from Pass {p_num + 1}...")
                    _download_btn(pass_ckpt_path)
                    break
                except Exception as e:
                    print(f"  ⚠️ Could not read {pass_ckpt_path}, checking earlier passes...")
                    
        for pass_num, pass_name, pass_fn in passes:
            if pass_num <= start_pass_idx:
                continue  # Already computed and loaded from disk
                
            t0 = time.time()
            print(f"  ▶ Running Pass {pass_num}/8: {pass_name}...")
            try:
                frames = []
                for tgt_df, lbl in [(state.train_s2_clean, "S2"), (state.train_s3_clean, "S3")]:
                    df_pass = pass_fn(s1_df, tgt_df, lbl)
                    if not df_pass.empty:
                        frames.append(df_pass)
                
                if frames:
                    pass_result = pd.concat(frames, ignore_index=True)
                    before_cnt = len(accumulated_df)
                    accumulated_df = pd.concat([accumulated_df, pass_result], ignore_index=True)
                    accumulated_df.drop_duplicates(subset=["source1_entity_id", "candidate_entity_id", "candidate_source"], inplace=True)
                    added = len(accumulated_df) - before_cnt
                    dt = time.time() - t0
                    print(f"    ✅ Pass {pass_num} Done in {dt:.1f}s — Added {added:,} new pairs | Total Unique Candidates: {len(accumulated_df):,}")
                else:
                    print(f"    ⚪ Pass {pass_num} produced 0 pairs ({time.time()-t0:.1f}s)")
                
                # 💾 Save checkpoint file to disk after EVERY pass!
                pass_ckpt_path = os.path.join(CKPT_DIR, f"{name_prefix}_candidates_pass{pass_num}.parquet")
                accumulated_df.to_parquet(pass_ckpt_path, index=False)
                _download_btn(pass_ckpt_path)
                gc.collect()
                
            except Exception as e:
                print(f"    ⚠️ Pass {pass_num} encountered an issue (e.g. RAM OOM): {e}")
                print(f"    📌 Preserving candidates up to Pass {pass_num-1} ({len(accumulated_df):,} candidates)")
                break
                
        return accumulated_df
        
    print("--- 🟢 Step A: Generating Training Candidates (Resumable) ---")
    state.train_candidates = run_pass_incremental_resumable(train_s1_df, "train")
    state.train_candidates.to_parquet(train_cands_ckpt, index=False)
    
    print("\n--- 🟢 Step B: Generating Validation Candidates (Resumable) ---")
    state.val_candidates = run_pass_incremental_resumable(val_s1_df, "val")
    state.val_candidates.to_parquet(val_cands_ckpt, index=False)
    
    _download_btn(train_cands_ckpt, val_cands_ckpt)

assert state.train_candidates is not None and state.val_candidates is not None
print(f"\n✅ Stage 4 Complete — Train cands: {len(state.train_candidates):,} | Val cands: {len(state.val_candidates):,}")


---
## 📊 Cell 7 — Stage 5: Evaluate Blocking Recall

In [ ]:
from pipeline import evaluate_candidate_recall

print("⏳ Stage 5: Computing blocking recall...")
evaluate_candidate_recall(cfg, state)
print("✅ Stage 5 Complete — Recall numbers printed in logs above.")


---
## 🏗️ Cell 8 — Stage 6: Build Training Pairs & Hard Negatives

> 💾 **Checkpoint:** Saves `train_pairs.parquet` and `val_pairs.parquet` to `/kaggle/working/output/`.
> A **⬇️ download button** will appear in this cell's output immediately after saving.
>
> **Re-run only if you changed:** negative sampling ratio (`neg_ratio`) or `hard_neg_fraction` in `config.py`.

In [ ]:
from pipeline import build_training_pairs

train_pairs_ckpt = os.path.join(CKPT_DIR, "train_pairs.parquet")
val_pairs_ckpt   = os.path.join(CKPT_DIR, "val_pairs.parquet")

if os.path.exists(train_pairs_ckpt) and os.path.exists(val_pairs_ckpt):
    print("⚡ [CHECKPOINT HIT] Loading training pairs from Kaggle disk — skipping hard-negative sampling...")
    state.train_pairs = pd.read_parquet(train_pairs_ckpt)
    state.val_pairs   = pd.read_parquet(val_pairs_ckpt)
    print(f"   Loaded train_pairs: {len(state.train_pairs):,} rows")
    print(f"   Loaded val_pairs  : {len(state.val_pairs):,} rows")
else:
    print("⏳ Stage 6: Labelling candidates & sampling hard negatives...")
    build_training_pairs(cfg, state)
    # ── Save to Kaggle disk ──────────────────────────────────────────────────
    state.train_pairs.to_parquet(train_pairs_ckpt, index=False)
    state.val_pairs.to_parquet(val_pairs_ckpt, index=False)
    sz_train = os.path.getsize(train_pairs_ckpt) / 1_048_576
    sz_val   = os.path.getsize(val_pairs_ckpt)   / 1_048_576
    print(f"   💾 Saved train_pairs.parquet ({sz_train:.1f} MB)")
    print(f"   💾 Saved val_pairs.parquet   ({sz_val:.1f} MB)")
    # ── Immediate download buttons appear here in cell output ────────────────
    _download_btn(train_pairs_ckpt, val_pairs_ckpt)

assert state.train_pairs is not None and state.val_pairs is not None
print(f"✅ Stage 6 Complete — Train pairs: {len(state.train_pairs):,} | Val pairs: {len(state.val_pairs):,}")


---
## 🧮 Cell 9 — Stage 7: Extract 30+ Pairwise Similarity Features

> 💾 **Checkpoint:** Saves `train_features.parquet` and `val_features.parquet` to `/kaggle/working/output/`.
> A **⬇️ download button** will appear in this cell's output immediately after saving.
> This is the **biggest time saver** — feature extraction takes ~15–25 minutes.
>
> **Re-run only if you changed:** `features.py` (added or removed feature columns).
> ✅ **Skip this cell** if you are only tuning XGBoost hyperparameters.

In [ ]:
import time, os, math
import pandas as pd
from features import extract_features, build_entity_lookup, get_feature_column_names

train_feat_ckpt = os.path.join(CKPT_DIR, "train_features.parquet")
val_feat_ckpt   = os.path.join(CKPT_DIR, "val_features.parquet")
feat_cols_ckpt  = os.path.join(CKPT_DIR, "feature_columns.txt")

if os.path.exists(train_feat_ckpt) and os.path.exists(val_feat_ckpt):
    print("⚡ [CHECKPOINT HIT] Loading features from Kaggle disk — skipping feature extraction...")
    state.train_merged = pd.read_parquet(train_feat_ckpt)
    state.val_merged   = pd.read_parquet(val_feat_ckpt)
    state.train_features = state.train_merged
    state.val_features   = state.val_merged
    if os.path.exists(feat_cols_ckpt):
        with open(feat_cols_ckpt) as f:
            state.feature_columns = [line.strip() for line in f if line.strip()]
    else:
        non_feat = {"source1_entity_id", "candidate_entity_id", "candidate_source", "is_match", "label", "source"}
        state.feature_columns = [c for c in state.train_merged.columns if c not in non_feat]
    print(f"   Loaded train_features: {state.train_merged.shape}")
    print(f"   Loaded val_features  : {state.val_merged.shape}")
    print(f"   Feature columns      : {len(state.feature_columns)} features")
else:
    print("⏳ Stage 7: Chunked & Incremental Feature Extraction starting...")
    
    entity_lookup = build_entity_lookup(
        state.train_s1_clean, state.train_s2_clean, state.train_s3_clean
    )
    
    CHUNK_SIZE = 500_000
    
    def extract_features_incremental(pairs_df, name_prefix):
        n_chunks = math.ceil(len(pairs_df) / CHUNK_SIZE)
        print(f"  ▶ Processing {len(pairs_df):,} pairs across {n_chunks} chunk(s) (Chunk Size: {CHUNK_SIZE:,})...")
        
        accumulated_chunks = []
        total_time = 0
        
        for chunk_idx in range(n_chunks):
            start_i = chunk_idx * CHUNK_SIZE
            end_i   = min((chunk_idx + 1) * CHUNK_SIZE, len(pairs_df))
            chunk_df = pairs_df.iloc[start_i:end_i]
            
            t0 = time.time()
            print(f"    ▶ Chunk {chunk_idx + 1}/{n_chunks} ({len(chunk_df):,} pairs)...", end="", flush=True)
            
            feats_df, _ = extract_features(pairs_df=chunk_df, entity_lookup=entity_lookup)
            
            if "is_match" in chunk_df.columns:
                feats_df["is_match"] = chunk_df["is_match"].values
            feats_df["source1_entity_id"] = chunk_df["source1_entity_id"].values
            feats_df["candidate_entity_id"] = chunk_df["candidate_entity_id"].values
            feats_df["candidate_source"] = chunk_df["candidate_source"].values
            
            dt = time.time() - t0
            total_time += dt
            print(f" Done in {dt:.1f}s!")
            
            accumulated_chunks.append(feats_df)
            partial_df = pd.concat(accumulated_chunks, ignore_index=True)
            
            # 💾 Save intermediate parquet & display download button
            chunk_ckpt = os.path.join(CKPT_DIR, f"{name_prefix}_features_chunk{chunk_idx + 1}.parquet")
            partial_df.to_parquet(chunk_ckpt, index=False)
            print(f"       💾 Saved {os.path.basename(chunk_ckpt)} ({len(partial_df):,} pairs processed so far)")
            _download_btn(chunk_ckpt)
            
        merged = pd.concat(accumulated_chunks, ignore_index=True)
        print(f"  ✅ {name_prefix.capitalize()} Feature Extraction Complete in {total_time/60:.1f} mins ({len(merged):,} pairs)")
        return merged
        
    print("--- 🟢 Step A: Train Features ---")
    state.train_merged = extract_features_incremental(state.train_pairs, "train")
    state.train_merged.to_parquet(train_feat_ckpt, index=False)
    
    print("\n--- 🟢 Step B: Validation Features ---")
    state.val_merged = extract_features_incremental(state.val_pairs, "val")
    state.val_merged.to_parquet(val_feat_ckpt, index=False)
    state.train_features = state.train_merged
    state.val_features   = state.val_merged
    
    non_feat = {"source1_entity_id", "candidate_entity_id", "candidate_source", "is_match", "label", "source"}
    state.feature_columns = [c for c in state.train_merged.columns if c not in non_feat]
    with open(feat_cols_ckpt, "w") as f:
        f.write("\n".join(state.feature_columns))
        
    _download_btn(train_feat_ckpt, val_feat_ckpt, feat_cols_ckpt)

assert state.feature_columns is not None
print(f"✅ Stage 7 Complete — {len(state.feature_columns)} features: {state.feature_columns[:6]}...")


---
## 🤖 Cell 10 — Stage 8: Train XGBoost (GPU)

> ❌ **No checkpoint skip here** — always re-trains when you run this cell.
> The trained model is saved as `model.json` automatically by the pipeline.

In [ ]:
from pipeline import train_xgboost

print("⏳ Stage 8: Training XGBoost classifier...")
try:
    train_xgboost(cfg, state)
except Exception as gpu_err:
    print(f"⚠️  GPU attempt failed ({gpu_err}). Falling back to CPU...")
    cfg.xgboost.device = "cpu"
    train_xgboost(cfg, state)

model_path = os.path.join(OUTPUT_DIR, "model.json")
print(f"✅ Stage 8 Complete — Model saved to: {state.model_path}")
# ── Immediate download button for model ──────────────────────────────────────
if os.path.exists(model_path):
    _download_btn(model_path)


---
## 📈 Cell 11 — Stage 9: Score Validation Set

In [ ]:
from pipeline import score_validation

print("⏳ Stage 9: Scoring validation candidates...")
score_validation(cfg, state)
assert state.val_probabilities is not None
print(f"✅ Stage 9 Complete — Probabilities for {len(state.val_probabilities):,} val candidates")


---
## 🎯 Cell 12 — Stage 10: Macro F0.5 Threshold Sweep

In [ ]:
from pipeline import optimize_threshold

print("⏳ Stage 10: Sweeping thresholds 0.05 → 0.95 for best Macro F0.5...")
optimize_threshold(cfg, state)
assert state.best_threshold is not None
print(f"✅ Stage 10 Complete — Best threshold: {state.best_threshold:.4f}")


---
## 🔎 Cell 13 — Stage 11: Error Analysis

In [ ]:
from pipeline import perform_error_analysis

print("⏳ Stage 11: Analysing FP / FN errors...")
perform_error_analysis(cfg, state)
print("✅ Stage 11 Complete — See logs above for FP / FN breakdown.")


---
## 🧪 Cell 14 — Stage 12: Generate Test Candidates (Pre-Cleaned Test Data)

> 💾 **Checkpoint:** Saves `test_candidates.parquet` to `/kaggle/working/output/`.
> A **⬇️ download button** will appear in this cell's output immediately after saving.
>
> **Re-run only if you changed:** blocking rules or `name_top_k` / `address_top_k` values.
> ✅ **Skip this cell** if only re-training the model or changing threshold.

In [ ]:
from inference import generate_test_candidates as _gen_test_cands
from io_utils import read_tsv

test_cands_ckpt = os.path.join(CKPT_DIR, "test_candidates.parquet")
nrows = cfg.runtime.debug_sample_size

print(f"⏳ Stage 12: Loading pre-cleaned test data{' (SANITY CHECK 50 ROWS)' if nrows else ''}...")
state.test_s1_clean = read_tsv(cfg.data.test_source1_path, nrows=nrows)
state.test_s2_clean = read_tsv(cfg.data.test_source2_path, nrows=nrows)
state.test_s3_clean = read_tsv(cfg.data.test_source3_path, nrows=nrows)
print(f"   test_S1: {len(state.test_s1_clean):,} | test_S2: {len(state.test_s2_clean):,} | test_S3: {len(state.test_s3_clean):,}")

if os.path.exists(test_cands_ckpt):
    print("⚡ [CHECKPOINT HIT] Loading test candidates from Kaggle disk — skipping test blocking...")
    state.test_candidates = pd.read_parquet(test_cands_ckpt)
    print(f"   Loaded test_candidates: {len(state.test_candidates):,} rows")
else:
    print("⏳ Stage 12: Generating test candidate pairs (blocking)...")
    bc = cfg.blocking
    blocking_config = {
        "name_top_k":             bc.name_top_k,
        "address_top_k":          bc.address_top_k,
        "min_name_similarity":    bc.min_name_similarity,
        "min_address_similarity": bc.min_address_similarity,
        "tfidf_analyzer":         bc.tfidf_analyzer,
        "tfidf_ngram_range":      list(bc.tfidf_ngram_range),
        "tfidf_min_df":           bc.tfidf_min_df,
        "tfidf_sublinear_tf":     bc.tfidf_sublinear_tf,
        "use_token_blocking":     True,
    }
    state.test_candidates = _gen_test_cands(
        state.test_s1_clean, state.test_s2_clean, state.test_s3_clean,
        blocking_config=blocking_config,
    )
    state.test_candidates.to_parquet(test_cands_ckpt, index=False)
    sz = os.path.getsize(test_cands_ckpt) / 1_048_576
    print(f"   💾 Saved test_candidates.parquet ({sz:.1f} MB)")
    _download_btn(test_cands_ckpt)

assert state.test_candidates is not None
print(f"✅ Stage 12 Complete — Test candidates: {len(state.test_candidates):,}")


---
## 🧮 Cell 15 — Stage 13: Build Test Features

> 💾 **Checkpoint:** Saves `test_features.parquet` to `/kaggle/working/output/`.
> A **⬇️ download button** will appear in this cell's output immediately after saving.
>
> **Re-run only if you changed:** `features.py` (added or removed feature columns).
> ✅ **Skip this cell** if only re-training the XGBoost model or changing threshold.

In [ ]:
import time, os, math
import pandas as pd
from inference import extract_test_features

test_feat_ckpt = os.path.join(CKPT_DIR, "test_features.parquet")

if os.path.exists(test_feat_ckpt):
    print("⚡ [CHECKPOINT HIT] Loading test features from Kaggle disk — skipping test feature extraction...")
    state.test_features = pd.read_parquet(test_feat_ckpt)
    print(f"   Loaded test_features: {state.test_features.shape}")
else:
    print("⏳ Stage 13: Chunked Test Feature Extraction starting...")
    
    CHUNK_SIZE = 500_000
    n_chunks = math.ceil(len(state.test_candidates) / CHUNK_SIZE)
    print(f"  ▶ Processing {len(state.test_candidates):,} test pairs across {n_chunks} chunk(s)...")
    
    accumulated_test_feats = []
    accumulated_test_meta  = []
    total_time = 0
    
    for chunk_idx in range(n_chunks):
        start_i = chunk_idx * CHUNK_SIZE
        end_i   = min((chunk_idx + 1) * CHUNK_SIZE, len(state.test_candidates))
        chunk_df = state.test_candidates.iloc[start_i:end_i]
        
        t0 = time.time()
        print(f"    ▶ Test Chunk {chunk_idx + 1}/{n_chunks} ({len(chunk_df):,} pairs)...", end="", flush=True)
        
        feats_df, meta_df = extract_test_features(
            test_candidates_df=chunk_df,
            test_s1_df=state.test_s1_clean,
            test_s2_df=state.test_s2_clean,
            test_s3_df=state.test_s3_clean,
        )
        
        dt = time.time() - t0
        total_time += dt
        print(f" Done in {dt:.1f}s!")
        
        accumulated_test_feats.append(feats_df)
        accumulated_test_meta.append(meta_df)
        
        partial_feats = pd.concat(accumulated_test_feats, ignore_index=True)
        
        # 💾 Save intermediate checkpoint & render download button
        chunk_ckpt = os.path.join(CKPT_DIR, f"test_features_chunk{chunk_idx + 1}.parquet")
        partial_feats.to_parquet(chunk_ckpt, index=False)
        print(f"       💾 Saved {os.path.basename(chunk_ckpt)} ({len(partial_feats):,} pairs processed so far)")
        _download_btn(chunk_ckpt)
        
    state.test_features = pd.concat(accumulated_test_feats, ignore_index=True)
    state.test_metadata = pd.concat(accumulated_test_meta, ignore_index=True)
    
    state.test_features.to_parquet(test_feat_ckpt, index=False)
    print(f"  ✅ Test Feature Extraction Complete in {total_time/60:.1f} mins ({state.test_features.shape})")
    _download_btn(test_feat_ckpt)

assert state.test_features is not None
print(f"✅ Stage 13 Complete — Test feature matrix: {state.test_features.shape}")


---
## 🎯 Cell 16 — Stage 14: Run Test Inference

In [ ]:
from pipeline import run_test_inference

print("⏳ Stage 14: Scoring test candidates with trained model...")
run_test_inference(cfg, state)
assert state.test_predictions is not None
n_matched = sum(1 for v in state.test_predictions.values() if v)
print(f"✅ Stage 14 Complete — {len(state.test_predictions):,} S1 entities | {n_matched:,} with matches")


---
## 📝 Cell 17 — Stage 15: Generate Submission TSV Files

> A **⬇️ download button** will appear immediately after the submission files are written.

In [ ]:
from pipeline import create_submission_files

print("⏳ Stage 15: Writing submission files...")
create_submission_files(cfg, state)

matching_path   = os.path.join(cfg.data.output_dir, "matching_results.tsv")
candidates_path = os.path.join(cfg.data.output_dir, "candidate_pairs.tsv")

print(f"\n🎉 Stage 15 Complete — Submission files in: {cfg.data.output_dir}")
print(f"   📄 {matching_path}")
print(f"   📄 {candidates_path}")
# ── Immediate download buttons appear here in cell output ────────────────────
_download_btn(matching_path, candidates_path)


---
## ✅ Cell 18: Inspect & Validate Submission Files

In [ ]:
import pandas as pd
from IPython.display import display

matching_path   = os.path.join(cfg.data.output_dir, "matching_results.tsv")
candidates_path = os.path.join(cfg.data.output_dir, "candidate_pairs.tsv")

if os.path.exists(matching_path) and os.path.exists(candidates_path):
    m_df = pd.read_csv(matching_path,   sep="\t", dtype=str, keep_default_na=False)
    c_df = pd.read_csv(candidates_path, sep="\t", dtype=str, keep_default_na=False)

    print("📊 matching_results.tsv")
    print(f"   Total S1 entities : {len(m_df):,}")
    print(f"   With matches      : {(m_df['matched_entity_ids'] != '').sum():,}")
    print(f"   Singletons        : {(m_df['matched_entity_ids'] == '').sum():,}")
    print()
    print("📊 candidate_pairs.tsv")
    print(f"   Total pairs       : {len(c_df):,}")
    print()
    print("First 10 rows of matching_results.tsv:")
    display(m_df.head(10))
else:
    print("⚠️  Submission files not found — ensure Cell 17 completed successfully.")


---
## 📦 Cell 19: Download ALL Checkpoints as One Zip

> Use this if you want **everything in a single archive** at the end.
> For individual file downloads, use the **⬇️ buttons that appeared in each cell above** — those are immediate and don't require waiting until here.

In [ ]:
import shutil

checkpoint_files = [
    "train_candidates.parquet",
    "val_candidates.parquet",
    "train_pairs.parquet",
    "val_pairs.parquet",
    "train_features.parquet",
    "val_features.parquet",
    "feature_columns.txt",
    "test_candidates.parquet",
    "test_features.parquet",
    "model.json",
    "matching_results.tsv",
    "candidate_pairs.tsv",
]

print("📂 Checkpoint files on Kaggle disk (/kaggle/working/output/):")
print("-" * 65)
total_size_mb = 0
found_files = []
for fname in checkpoint_files:
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(fpath):
        mb = os.path.getsize(fpath) / 1_048_576
        total_size_mb += mb
        found_files.append(fpath)
        print(f"  ✅ {fname:<35s}  {mb:7.1f} MB")
    else:
        print(f"  ⬜ {fname:<35s}  (not yet generated)")
print("-" * 65)
print(f"  Total: {len(found_files)} files  |  {total_size_mb:.1f} MB")

if found_files:
    zip_path = "/kaggle/working/checkpoints_and_outputs"
    print(f"\n⏳ Creating zip archive...")
    shutil.make_archive(base_name=zip_path, format="zip", root_dir=OUTPUT_DIR)
    zip_size_mb = os.path.getsize(zip_path + ".zip") / 1_048_576
    print(f"✅ Zip created ({zip_size_mb:.1f} MB)")
    # ── Immediate download button for the full zip ───────────────────────────
    _download_btn(zip_path + ".zip")
else:
    print("\n⚠️ No checkpoint files found yet. Run the pipeline cells above first.")
